In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib ipympl

In [ ]:
# ── User-configurable parameters ──────────────────────────────────────────
step_size     = 0.3   # leapfrog step size ε
max_tree_depth = 5    # maximum binary-tree depth (caps trajectory length)
repeated_times = 20   # number of NUTS iterations to run

# Choose distribution: "multimodal normal"  or  "singlemodal normal"
# If you supply your own, define log_prob(x,y) and grad_log_prob(x,y) below.
distribution = "multimodal normal"

In [ ]:
if distribution == "multimodal normal":
    sqrt3 = np.sqrt(3)
    K = 1 / (2.4 * np.pi)
    log_K = np.log(K)
    def log_prob(x, y):
        A = np.exp(-2.5 * ((x - 1.5*sqrt3)**2 + (y - 1.5)**2))
        B = np.exp(-2.5 * ((x + 1.5*sqrt3)**2 + (y - 1.5)**2))
        C = np.exp(-2.5 * (x**2 + (y + 3)**2))
        return log_K + np.log(A + B + C)
    def grad_log_prob(x, y):
        A = np.exp(-2.5 * ((x - 1.5*sqrt3)**2 + (y - 1.5)**2))
        B = np.exp(-2.5 * ((x + 1.5*sqrt3)**2 + (y - 1.5)**2))
        C = np.exp(-2.5 * (x**2 + (y + 3)**2))
        s = A + B + C
        dx = (-5 / s) * (A*(x - 1.5*sqrt3) + B*(x + 1.5*sqrt3) + C*x)
        dy = (-5 / s) * (A*(y - 1.5)        + B*(y - 1.5)        + C*(y + 3))
        return np.array([dx, dy])
elif distribution == "singlemodal normal":
    def log_prob(x, y):
        return -0.5 * (x**2 + y**2)
    def grad_log_prob(x, y):
        return np.array([-x, -y])
else:
    raise ValueError(f"Unknown distribution: {distribution}")

In [ ]:
x_range = np.linspace(-5, 5, 500)
y_range = np.linspace(-5, 5, 500)
X, Y = np.meshgrid(x_range, y_range)
Z = np.exp(log_prob(X, Y))

fig = plt.figure(figsize=(11, 5.5))
ax   = fig.add_subplot(121, projection='3d')
ax.plot_surface(X, Y, Z, cmap='Blues', alpha=0.5, edgecolor='none')
ax.set_xlabel('X', fontsize=10)
ax.set_ylabel('Y', fontsize=10)
ax.set_zlabel('Probability Density', fontsize=10)
ax.set_title('NUTS 3D Visualization', fontsize=12)
ax.view_init(elev=20, azim=45)
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_zlim(0, 0.15)

ax2d = fig.add_subplot(122)
ax2d.contour(X, Y, Z, cmap='Blues', alpha=0.5)
ax2d.set_xlabel('X', fontsize=10)
ax2d.set_ylabel('Y', fontsize=10)
ax2d.set_title('NUTS 2D Visualization', fontsize=12)
ax2d.set_xlim(-5, 5)
ax2d.set_ylim(-5, 5)
ax2d.grid(alpha=0.3)

In [ ]:
last_q = np.random.uniform(low=-3, high=3, size=2).astype(np.float64)

# Plot handles (cleared each iteration)
trajectory_line    = None
current_point      = None
trajectory_line_2d = None
current_point_2d   = None
result_text        = None

In [ ]:
def leapfrog(q, p, direction):
    """One leapfrog step in the given direction (+1 forward, -1 backward)."""
    eps = direction * step_size
    p_half = p + 0.5 * eps * grad_log_prob(*q)
    q_new  = q + eps * p_half
    p_new  = p_half + 0.5 * eps * grad_log_prob(*q_new)
    return q_new, p_new

def hamiltonian(q, p):
    """H = -log π(q) + 0.5 ||p||²"""
    return -log_prob(*q) + 0.5 * np.dot(p, p)

In [ ]:
def build_tree(q, p, u, direction, depth, H0, all_points):
    """
    Recursive NUTS tree builder (Algorithm 3 from Hoffman & Gelman 2014).

    Parameters
    ----------
    q, p        : current position / momentum at the leaf
    u           : slice variable  (this is log_u = log(Uniform(0,1)) - H0)
    direction   : +1 (forward) or -1 (backward)
    depth       : remaining tree depth to build
    H0          : Hamiltonian at the initial point of this iteration
    all_points  : list to collect every visited q (for visualisation)

    Returns
    -------
    q_minus, p_minus  : leftmost leaf
    q_plus,  p_plus   : rightmost leaf
    q_prime           : proposed sample from this subtree
    n_prime           : number of valid (slice-accepted) points
    s_prime           : 1 if no U-turn detected, else 0
    alpha_prime       : sum of min(1, exp(H0 - H)) for MH criterion
    n_alpha           : number of leapfrog steps taken
    """
    if depth == 0:
        # Base case: single leapfrog step
        q_new, p_new = leapfrog(q, p, direction)
        H_new = hamiltonian(q_new, p_new)
        all_points.append(q_new.copy())

        n_prime     = int(u <= -H_new)           # slice criterion  (u is already log_u)
        s_prime     = int(-H_new > u - 1000)     # divergence guard
        alpha_prime = min(1.0, np.exp(H0 - H_new))
        return q_new, p_new, q_new, p_new, q_new, n_prime, s_prime, alpha_prime, 1

    # Recursive case: build two subtrees
    (q_minus, p_minus, q_plus, p_plus,
     q_prime, n_prime, s_prime,
     alpha_prime, n_alpha) = build_tree(
         q, p, u, direction, depth - 1, H0, all_points)

    if s_prime == 1:
        if direction == -1:
            (q_minus, p_minus, _, _,
             q_double_prime, n_double_prime, s_double_prime,
             alpha_double_prime, n_alpha2) = build_tree(
                 q_minus, p_minus, u, direction, depth - 1, H0, all_points)
        else:
            (_, _, q_plus, p_plus,
             q_double_prime, n_double_prime, s_double_prime,
             alpha_double_prime, n_alpha2) = build_tree(
                 q_plus, p_plus, u, direction, depth - 1, H0, all_points)

        # Metropolis-style selection between the two subtree proposals
        if n_prime + n_double_prime > 0:
            accept_ratio = n_double_prime / (n_prime + n_double_prime)
        else:
            accept_ratio = 0.0
        if np.random.rand() < accept_ratio:
            q_prime = q_double_prime

        # No-U-Turn criterion
        delta_q = q_plus - q_minus
        s_prime = (s_double_prime
                   * int(np.dot(delta_q, p_minus) >= 0)
                   * int(np.dot(delta_q, p_plus)  >= 0))

        n_prime     += n_double_prime
        alpha_prime += alpha_double_prime
        n_alpha     += n_alpha2

    return (q_minus, p_minus, q_plus, p_plus,
            q_prime, n_prime, s_prime, alpha_prime, n_alpha)

In [ ]:
def run_nuts():
    global last_q, trajectory_line, current_point
    global trajectory_line_2d, current_point_2d, result_text

    q0 = last_q.copy()
    p0 = np.random.randn(2)          # fresh momentum
    H0 = hamiltonian(q0, p0)

    # Slice variable: log u ~ Uniform(-H0, 0)  →  u ~ Uniform(0, exp(-H0))
    log_u = np.log(np.random.rand()) - H0   # equivalent to log(Uniform(0,1)) + (-H0)

    # Initialise tree
    q_minus = q0.copy();  p_minus = p0.copy()
    q_plus  = q0.copy();  p_plus  = p0.copy()
    q_prime = q0.copy()
    n = 1          # number of valid points so far
    s = 1          # stop flag
    depth = 0

    all_points = [q0.copy()]   # collect trajectory for visualisation

    # ── Clear previous trajectory ──────────────────────────────────────────
    for handle in [trajectory_line, current_point,
                   trajectory_line_2d, current_point_2d]:
        if handle is not None:
            handle.remove()
    trajectory_line = current_point = None
    trajectory_line_2d = current_point_2d = None

    # ── Grow the tree ──────────────────────────────────────────────────────
    while s == 1 and depth < max_tree_depth:
        direction = 1 if np.random.rand() > 0.5 else -1

        if direction == -1:
            (q_minus, p_minus, _, _,
             q_double_prime, n_prime, s_prime,
             alpha_prime, n_alpha) = build_tree(
                 q_minus, p_minus, log_u, direction, depth, H0, all_points)
        else:
            (_, _, q_plus, p_plus,
             q_double_prime, n_prime, s_prime,
             alpha_prime, n_alpha) = build_tree(
                 q_plus, p_plus, log_u, direction, depth, H0, all_points)

        # Biased progressive sampling
        if s_prime == 1 and n > 0:
            if np.random.rand() < n_prime / n:
                q_prime = q_double_prime

        # No-U-Turn check on the full tree
        delta_q = q_plus - q_minus
        s = (s_prime
             * int(np.dot(delta_q, p_minus) >= 0)
             * int(np.dot(delta_q, p_plus)  >= 0))

        n += n_prime
        depth += 1

        # ── Animate the current tree state ─────────────────────────────────
        pts = np.array(all_points)
        z_vals = np.exp(log_prob(pts[:, 0], pts[:, 1]))

        if trajectory_line:    trajectory_line.remove()
        if current_point:      current_point.remove()
        if trajectory_line_2d: trajectory_line_2d.remove()
        if current_point_2d:   current_point_2d.remove()

        trajectory_line = ax.plot(
            pts[:, 0], pts[:, 1], z_vals,
            color='#FF4500', marker='o', markersize=6, linewidth=3)[0]
        current_point = ax.scatter(
            q_prime[0], q_prime[1],
            np.exp(log_prob(q_prime[0], q_prime[1])),
            color='#FFFF00', s=200, zorder=100,
            edgecolor='black', linewidth=1)
        trajectory_line_2d = ax2d.plot(
            pts[:, 0], pts[:, 1],
            color='#FF4500', marker='o', markersize=6, linewidth=3)[0]
        current_point_2d = ax2d.scatter(
            q_prime[0], q_prime[1],
            color='#FFFF00', s=200, zorder=100,
            edgecolor='black', linewidth=1)

        fig.canvas.draw()
        fig.canvas.flush_events()
        time.sleep(0.15)

    # ── Accept / reject ────────────────────────────────────────────────────
    H_prime = hamiltonian(q_prime, p0)   # use original p0 for MH ratio
    accept_prob = min(1.0, np.exp(H0 - H_prime))

    if np.random.rand() < accept_prob:
        result = f"Accept!  P: {accept_prob:.2f}  depth: {depth}"
        color  = 'green'
        last_q = q_prime.copy()
    else:
        result = f"Deny!    P: {accept_prob:.2f}  depth: {depth}"
        color  = 'red'
        # last_q stays at q0

    final_q = last_q.copy()
    z_final = np.exp(log_prob(final_q[0], final_q[1]))
    ax.scatter(final_q[0], final_q[1], z_final,
               color=color, s=100, alpha=0.5, zorder=50)
    ax2d.scatter(final_q[0], final_q[1],
                 color=color, s=100, alpha=0.5, zorder=50)

    if result_text:
        result_text.remove()
    result_text = ax.text2D(
        0.05, 0.95, result, transform=ax.transAxes,
        fontsize=11, color=color,
        bbox=dict(boxstyle='round', facecolor='white'))
    fig.canvas.draw()

In [ ]:
for _ in range(repeated_times):
    run_nuts()